# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')} | License: {getattr(metadata, 'license', 'N/A')}")
print("\nKeywords:", getattr(metadata, 'keywords', []))
print(f"Cite as: {getattr(metadata, 'citeAs', 'N/A')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id
print("Available Record Sets (by @id):\n")
record_sets = []

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # recordSet may be a list or a dict
    if isinstance(metadata.recordSet, list):
        for rs in metadata.recordSet:
            # Each rs is an object with @id
            rs_id = getattr(rs, '@id', None)
            rs_name = getattr(rs, 'name', rs_id)
            print(f"- {rs_id}: {rs_name}")
            record_sets.append(rs_id)
    else:
        rs = metadata.recordSet
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', rs_id)
        print(f"- {rs_id}: {rs_name}")
        record_sets.append(rs_id)
else:
    print("No record sets found with metadata.recordSet. Attempting to infer from data files...")
    # Fallback method: Try to infer via dataset.get_record_sets()

    try:
        # get_record_sets gives a dict mapping @id to RecordSet
        for rs_id, recordset in dataset.get_record_sets().items():
            print(f"- {rs_id}: {getattr(recordset, 'name', rs_id)}")
            record_sets.append(rs_id)
    except Exception as e:
        print("Could not retrieve record sets:", e)

# If at least one record set, print its fields
if record_sets:
    print("\nFields (@id) for the **first** record set:")
    rs_id = record_sets[0]
    record_sets_dict = dataset.get_record_sets()
    rs_obj = record_sets_dict[rs_id]
    fields = getattr(rs_obj, 'field', [])
    if fields:
        for field in fields:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', field_id)
            print(f"- {field_id}: {field_name}")
    else:
        print("No fields found in this record set.")
else:
    print("No record sets available in this dataset.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, let's extract all record sets into DataFrames.
# We use only the available record_sets found previously.

# If no record sets were found in the previous cell, try again
if not record_sets:
    record_sets = list(dataset.get_record_sets().keys())

dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet: {record_set_id}")
            print("Columns:", df.columns.tolist())
        else:
            print(f"No records found in RecordSet: {record_set_id}")
    except Exception as e:
        print(f"Failed to load RecordSet {record_set_id}: {str(e)}")

# Show the head of the largest DataFrame loaded
if dataframes:
    # Pick the record set with the most rows
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"\nPreview of records in record set '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
# We select the main DataFrame for exploration
if not dataframes:
    print('No dataframes available for analysis.')
else:
    df = dataframes[main_record_set_id]
    print(f"Columns: {df.columns.tolist()}")
    # Attempt to guess a numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not numeric_candidates:
        # Try converting columns to numeric if possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    # Choose one numeric field, fallback to None
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Filter: show records with values greater than threshold (mean, or 10 if all vals > 10)
        try:
            threshold = df[numeric_field].mean() if df[numeric_field].mean() > 10 else 10
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        except Exception as e:
            print(f"Could not filter and normalize numeric field '{numeric_field}': {e}")
        # Attempt to group by next available field if possible
        group_candidates = [col for col in df.columns if col != numeric_field and df[col].nunique() < len(df)/2]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field '{group_field}':")
            try:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                display(grouped_df.head())
            except Exception as e:
                print(f"Could not group by '{group_field}': {e}")
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[main_record_set_id]
    # Try to visualize the distribution of the numeric field
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
        # If group_field was found, boxplot by group
        if 'group_field' in locals() and group_field in df.columns:
            plt.figure(figsize=(10,6))
            sns.boxplot(x=df[group_field], y=df[numeric_field])
            plt.title(f"{numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
    else:
        print("No numeric field visualized.")
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the Croissant schema, inspected available record sets and fields using their `@id`s.
- Data from the largest record set was extracted and examined with basic statistics and visualized.
- The dataset contains complex proteomics results, with fields such as protein identifiers, abundance, and sample information.
- Further domain-specific analysis is possible by leveraging the full metadata (e.g., by examining modifications or comparing sample groups).